In [11]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [12]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


1469

In [13]:
# =======================================================================================
#
# BLOCK 3: TRAIN A NEW LIGHTGBM MEAN MODEL
#
# =======================================================================================
import lightgbm as lgb

N_OPTUNA_TRIALS = 100 # A strong number for a comprehensive search
# --- STAGE 1, PART 1: Tune Hyperparameters for the LightGBM Mean Model ---
print("\n--- STAGE 1, PART 1: Tuning a LightGBM model to predict the mean sale_price ---")

# Define the objective function for Optuna to minimize RMSE
def create_lgbm_objective(X_features, y_target):
    # Use a single split for faster tuning
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_target, test_size=0.25, random_state=RANDOM_STATE)

    def objective(trial):
        # Define the search space for LightGBM
        params = {
            'objective': 'regression_l1',
            'metric': 'rmse',
            'n_estimators': 3000, # Increased estimators to allow smaller learning rates to converge
            'n_jobs': -1,
            'verbose': -1,
            'seed': RANDOM_STATE,

            # Focused search on more promising values based on the last run
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 80, 250), # Expanded upwards
            
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.8), # Narrowed
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.8, 1.0), # Narrowed to higher values
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),

            'lambda_l1': trial.suggest_float('lambda_l1', 1e-6, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-6, 10.0, log=True),

            'min_child_samples': trial.suggest_int('min_child_samples', 20, 80), # Focused
        }
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        return rmse

    return objective

# Create and run the study
study_lgbm = optuna.create_study(direction='minimize')

# THE FIX IS HERE: Use study.optimize() instead of study.run()
study_lgbm.optimize(create_lgbm_objective(X, y_true), n_trials=N_OPTUNA_TRIALS)

# Store the best parameters
best_params_lgbm = study_lgbm.best_params
print("\n--- LightGBM Mean Model Tuning Complete ---")
print(f"Best validation RMSE found: ${study_lgbm.best_value:,.2f}")
print("Best hyperparameters for the LightGBM mean model:")
print(best_params_lgbm)

[I 2025-07-16 16:53:18,307] A new study created in memory with name: no-name-2d3e97c2-172e-4e20-9210-0aa7e355819f



--- STAGE 1, PART 1: Tuning a LightGBM model to predict the mean sale_price ---


[I 2025-07-16 16:53:51,939] Trial 0 finished with value: 102884.04348110095 and parameters: {'learning_rate': 0.05252029255460783, 'num_leaves': 167, 'feature_fraction': 0.6063955346078136, 'bagging_fraction': 0.9915195578147762, 'bagging_freq': 3, 'lambda_l1': 0.0009576900945182566, 'lambda_l2': 6.25728050798939e-05, 'min_child_samples': 29}. Best is trial 0 with value: 102884.04348110095.
[I 2025-07-16 16:54:19,123] Trial 1 finished with value: 106638.69586483775 and parameters: {'learning_rate': 0.01386292703668459, 'num_leaves': 90, 'feature_fraction': 0.5237760825135213, 'bagging_fraction': 0.9255622708371903, 'bagging_freq': 3, 'lambda_l1': 5.894250613241884e-05, 'lambda_l2': 0.09527497663478247, 'min_child_samples': 72}. Best is trial 0 with value: 102884.04348110095.
[I 2025-07-16 16:54:55,979] Trial 2 finished with value: 103155.87918025345 and parameters: {'learning_rate': 0.03133691157574653, 'num_leaves': 169, 'feature_fraction': 0.6717537876256399, 'bagging_fraction': 0.93

KeyboardInterrupt: 

In [ ]:
# --- STAGE 1, PART 2: K-Fold Training of the Tuned LightGBM Mean Model ---
print("\n--- STAGE 1, PART 2: K-Fold Training with Newly Tuned Parameters ---")

# Add fixed parameters to our newly tuned ones
final_params_lgbm = best_params_lgbm.copy()
final_params_lgbm.update({'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 2000,
                          'seed': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1})

# Initialize prediction arrays
oof_lgbm_preds = np.zeros(len(X))
test_lgbm_preds = np.zeros(len(X_test))

# Use the same K-Fold splits for consistency
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Training LightGBM Mean Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]

    model = lgb.LGBMRegressor(**final_params_lgbm)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    
    oof_lgbm_preds[val_idx] = model.predict(X_val)
    test_lgbm_preds += model.predict(X_test) / N_SPLITS

# --- Performance of the New LightGBM Model ---
final_mean_rmse_lgbm = np.sqrt(mean_squared_error(y_true, oof_lgbm_preds))
print(f"\nLightGBM Mean Model Final OOF RMSE: ${final_mean_rmse_lgbm:,.2f}")

In [ ]:
# =======================================================================================
#
# BLOCK 4: THE MEAN MODEL SHOWDOWN - RANKING THE MODELS
#
# =======================================================================================

# ASSUMPTION: You have the OOF predictions from your other models loaded.
# If not, you will need to load them from .npy files or re-run those notebooks.
# oof_mean_preds = ... (XGBoost OOF predictions)
# oof_catboost_preds = ... (CatBoost OOF predictions)

# Calculate the RMSE for each model
rmse_xgb = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
rmse_cb = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
rmse_lgbm = final_mean_rmse_lgbm # Already calculated in the previous cell

# Create a dictionary to hold the results
model_scores = {
    'XGBoost': rmse_xgb,
    'CatBoost': rmse_cb,
    'LightGBM': rmse_lgbm
}

# Sort the models by their RMSE score (lower is better)
sorted_models = sorted(model_scores.items(), key=lambda item: item[1])

print("\n" + "="*50)
print("         MEAN MODEL ACCURACY RANKING (OOF RMSE)")
print("="*50)

for i, (model_name, score) in enumerate(sorted_models):
    print(f"  Rank {i+1}: {model_name:<10} - ${score:,.2f}")

print("="*50)
print("\nThese three models are now ready to be used in your ensemble pipeline.")